# Week 12: Hybrid search, significance testing, and a reproducible CLI (milestone M4)

| Session | Topic | Time |
|---|---|---|
| Day 1 | Score fusion: min-max + linear fusion, reciprocal rank fusion | 2 h |
| Day 2 | Is the difference real? Paired tests across queries | 2 h |
| Day 3 | Error analysis: read the failures | 1.5 h |
| Day 4 | Make it reproducible: a YAML config + command-line evaluation | 2 h |
| Check | **Cumulative check 3**: implement an unfamiliar ranking model (60 min) | 1 h |
| Buffer | Catch up on Weeks 9–12 | 1.5 h |

### Milestone M4
- [ ] `medlab/fusion.py` and `medlab/stats.py`, with tests.
- [ ] Experiment A's first results table: BM25, dense and hybrid, with 95% CIs and p-values versus BM25.
- [ ] `python -m medlab.evaluate --config configs/bm25.yaml` reproduces a row of that table from a clean terminal.
- [ ] `ERROR_ANALYSIS.md` with 5 annotated failure cases.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

In [ ]:
import json
import numpy as np
import pandas as pd
from course.data import load_nfcorpus
from course.embeddings import encode
index_mod, text_mod, ranking, metrics, dense = (fresh(m) for m in ["medlab.index", "medlab.text", "medlab.ranking", "medlab.metrics", "medlab.dense"])
pp = text_mod.preprocess

def load_split(split):
    corpus, queries, qrels = load_nfcorpus(split)
    return queries, qrels

corpus, _, _ = load_nfcorpus("test")
doc_ids = list(corpus)
index = index_mod.InvertedIndex.build({d: v["title"] + " " + v["text"] for d, v in corpus.items()}, pp)
doc_emb = encode([corpus[d]["title"] + " " + corpus[d]["text"] for d in doc_ids], cache_name="nfcorpus_minilm")

def bm25_run(queries, k1=1.2, b=0.75):
    r = ranking.BM25(index, k1, b)
    return {q: dict(r.search(pp(t), k=100)) for q, t in queries.items()}

def dense_run(queries):
    qids = list(queries)
    return dense.dense_search(doc_emb, doc_ids, encode([queries[q] for q in qids]), qids, k=100)

dev_q, dev_qrels = load_split("dev"); test_q, test_qrels = load_split("test")
runs_dev = {"bm25": bm25_run(dev_q), "dense": dense_run(dev_q)}
runs_test = {"bm25": bm25_run(test_q), "dense": dense_run(test_q)}

---
# Day 1: Fusion (2 h)

BM25 and dense retrieval make **different mistakes**: lexical matching catches exact terms (drug names, gene symbols), while dense matching catches paraphrases. Combining them is often better than either alone.

**Linear fusion:** the two scores live on different scales (BM25 ≈ 0–30, cosine ≈ 0–1), so **min-max normalise** each query's scores to [0, 1] first, then compute $\alpha \cdot s_{\text{bm25}} + (1 - \alpha) \cdot s_{\text{dense}}$. A document missing from one run gets 0 there.

**Reciprocal rank fusion (RRF):** ignore the scores and use only the ranks: $\text{RRF}(d) = \sum_{\text{runs}} \frac{1}{k + \text{rank}(d)}$, with $k = 60$. It has no scale problems and no weight to tune, and it's surprisingly hard to beat.

### Exercise 12.1: `capstone/medlab/fusion.py`
```python
minmax(scores: dict) -> dict                       # all-equal scores → all 1.0
linear_fusion(run_a, run_b, alpha) -> run          # per query, over the union of docs
rrf(runs: list[run], k=60) -> run                  # rank 1 = best (score desc, doc_id asc)
```
Write `tests/test_fusion.py` with at least 4 hand-computed cases.

In [ ]:
# Implement fusion.py and tests, then run the checker.

In [ ]:
fusion = fresh("medlab.fusion")
assert run_tests("tests/test_fusion.py") == 0
assert fusion.minmax({"a": 2.0, "b": 4.0}) == {"a": 0.0, "b": 1.0}
assert fusion.minmax({"a": 3.0}) == {"a": 1.0}
r = fusion.rrf([{"q": {"a": 9, "b": 5}}, {"q": {"b": 0.9, "c": 0.1}}], k=60)
assert abs(r["q"]["b"] - (1 / 62 + 1 / 61)) < 1e-12 and abs(r["q"]["c"] - 1 / 62) < 1e-12
lf = fusion.linear_fusion({"q": {"a": 10, "b": 0}}, {"q": {"b": 1.0, "c": 0.0}}, alpha=0.5)
assert lf["q"] == {"a": 0.5, "b": 0.5, "c": 0.0}

alphas = np.linspace(0, 1, 11)
dev_scores = [metrics.evaluate_run(fusion.linear_fusion(runs_dev["bm25"], runs_dev["dense"], a), dev_qrels)["ndcg@10"] for a in alphas]
best_alpha = float(alphas[int(np.argmax(dev_scores))])
print("best alpha on dev:", best_alpha)
runs_test["hybrid_linear"] = fusion.linear_fusion(runs_test["bm25"], runs_test["dense"], best_alpha)
runs_test["hybrid_rrf"] = fusion.rrf([runs_test["bm25"], runs_test["dense"]])
print("✅ 12.1 passed")

---
# Day 2: Are the differences real? (2 h)

In IR, the **unit of analysis is the query**. Compute nDCG@10 *per query* for systems A and B, and look at the per-query differences $d_q$. Two tools, both from Week 8, now applied to paired per-query scores:
- **Paired randomisation test:** under H₀, the sign of each $d_q$ is a coin flip. Flip signs at random 10,000 times; the p-value is the share of $|\text{mean}|$ values at least as large as the observed one.
- **Bootstrap CI of the mean difference:** resample *queries* with replacement.

**Multiple comparisons:** you're about to compare 3 systems against BM25. With 3 tests at α = 0.05, the chance of at least one false "win" is about 14%. Use the **Holm–Bonferroni** correction: sort the p-values from smallest to largest, and compare the *i*-th smallest with α/(m − i + 1).

### Exercise 12.2: `capstone/medlab/stats.py`
```python
paired_randomization_test(a, b, n_perm=10000, seed=0) -> p_value     # a, b: per-query score arrays
bootstrap_mean_diff_ci(a, b, n_boot=10000, alpha=0.05, seed=0) -> (low, high)
holm(p_values: dict[str, float], alpha=0.05) -> dict[str, bool]      # True = significant
```

In [ ]:
# Implement stats.py and tests/test_stats.py, then run the checker.

In [ ]:
stats = fresh("medlab.stats")
assert run_tests("tests/test_stats.py") == 0
rng = np.random.default_rng(0)
a = rng.random(50); assert stats.paired_randomization_test(a, a + 0.2) < 0.001
assert stats.paired_randomization_test(a, a) == 1.0
lo, hi = stats.bootstrap_mean_diff_ci(a + 0.2 + 0.1 * rng.normal(size=50), a); assert lo < 0.2 < hi
assert stats.holm({"x": 0.01, "y": 0.04, "z": 0.03}) == {"x": True, "y": False, "z": False}

pq = {name: metrics.per_query(run, test_qrels) for name, run in runs_test.items()}
qids = sorted(test_qrels)
score = lambda name: np.array([pq[name][q]["ndcg@10"] for q in qids])
rows, pvals = [], {}
for name in runs_test:
    row = {"system": name, "nDCG@10": score(name).mean()}
    if name != "bm25":
        row["Δ vs BM25"] = score(name).mean() - score("bm25").mean()
        row["95% CI"] = "[{:+.3f}, {:+.3f}]".format(*stats.bootstrap_mean_diff_ci(score(name), score("bm25")))
        pvals[name] = row["p"] = stats.paired_randomization_test(score(name), score("bm25"))
    rows.append(row)
sig = stats.holm(pvals)
table = pd.DataFrame(rows).set_index("system"); table["significant (Holm)"] = pd.Series(sig)
print(table.round(4))
table.round(4).to_csv("../capstone/results/week12_table_A.csv")
print("✅ 12.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Randomisation: `d = a - b`, `obs = abs(d.mean())`, `signs = rng.choice([-1, 1], size=(n_perm, len(d)))`, then `p = (np.sum(np.abs((signs * d).mean(axis=1)) >= obs - 1e-12) + 1) / (n_perm + 1)`. When `d` is all zeros, every permutation ties, so p = 1.

</details>

<details><summary>Hint 2</summary>

Holm: sort the items by p. Walk through them; at step i (from 0), compare with `alpha / (m - i)`. Stop at the first failure: everything after it is not significant either.

</details>


✍️ *Write the result sentence for your report: 'Hybrid (RRF) improved nDCG@10 over BM25 by … (95% CI […]), p = …, Holm-corrected …'. Is any improvement both statistically significant and *practically* meaningful?* (double-click to edit)



---
# Day 3: Error analysis (1.5 h)

Metrics tell you *how much*. Reading failures tells you *why*, and why is what gives you ideas for improvements and limitations for your report.

In [ ]:
diff = pd.Series({q: pq["dense"][q]["ndcg@10"] - pq["bm25"][q]["ndcg@10"] for q in qids}).sort_values()
print("BM25 wins most:"); print(diff.head(3))
print("Dense wins most:"); print(diff.tail(3))
q = diff.index[0]
print("\nQUERY:", test_q[q])
top_bm25 = sorted(runs_test["bm25"][q], key=runs_test["bm25"][q].get, reverse=True)[:3]
for d in top_bm25:
    print(f"  BM25 top: [{test_qrels[q].get(d, 0)}] {corpus[d]['title'][:90]}")

### Exercise 12.3: `capstone/ERROR_ANALYSIS.md`
Analyse **5 queries**: 2 where BM25 wins clearly, 2 where dense wins, and 1 where both fail. For each, write the query, what each system returned, and a **category**: *vocabulary mismatch*, *ambiguous query*, *judgement gap* (a relevant-looking document that was never judged), *over-matching a frequent term*, or your own. End with a summary: which category is most common, and what would fix it?

In [ ]:
# Write ERROR_ANALYSIS.md in VS Code, then run the checker.

In [ ]:
txt = (CAP / "ERROR_ANALYSIS.md").read_text()
assert txt.lower().count("category") >= 5, "label each of the 5 cases with a category"
print("✅ 12.3 passed")

---
# Day 4: A reproducible command-line evaluation (2 h)

A result that only exists in a notebook cell isn't reproducible. Real projects have **scripts driven by config files**, so anybody can rerun them exactly.

**First, make the capstone stand alone.** It mustn't import the course's `course/` folder:
1. Copy `ai-programme/course/data.py` to `capstone/medlab/data.py` (keep `load_nfcorpus` and the fake-data parts you need).
2. Copy `course/embeddings.py` to `capstone/medlab/embeddings.py`, changing its import to `from medlab.data import DATA_DIR, fake_mode`.

### Exercise 12.4: `capstone/medlab/evaluate.py` and `capstone/configs/*.yaml`
```yaml
# configs/bm25.yaml
name: bm25_default
system: bm25            # one of: tfidf, bm25, dense, hybrid_rrf
split: test
params: {k1: 1.2, b: 0.75}
```
`python -m medlab.evaluate --config configs/bm25.yaml` must:
1. load the config (`yaml.safe_load`), the data, and build the system,
2. compute the run, the metrics and the per-query metrics,
3. write `results/<name>.json` with `{"config": …, "metrics": …, "per_query": …}`,
4. print the metrics,
5. use `argparse`, and have a `main(argv=None)` function so tests can call it.

Create configs for `bm25`, `dense` and `hybrid_rrf`.

In [ ]:
# Create medlab/data.py, medlab/embeddings.py, medlab/evaluate.py and the configs, then run the checker.

In [ ]:
import subprocess, sys, os
out = subprocess.run([sys.executable, "-m", "medlab.evaluate", "--config", "configs/bm25.yaml"], cwd=CAP, capture_output=True, text=True)
print(out.stdout[-800:], out.stderr[-1500:])
assert out.returncode == 0
res = json.load(open(CAP / "results" / "bm25_default.json"))
assert {"config", "metrics", "per_query"} <= set(res)
assert abs(res["metrics"]["ndcg@10"] - metrics.evaluate_run(runs_test["bm25"], test_qrels)["ndcg@10"]) < 1e-9
assert "course" not in (CAP / "medlab" / "evaluate.py").read_text()
for cfg in ["dense", "hybrid_rrf"]:
    assert (CAP / "configs" / f"{cfg}.yaml").exists()
print("✅ 12.4 passed (milestone M4)")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Structure: `def build_run(cfg, corpus, queries) -> run`, using `if cfg['system'] == ...` branches, then `def main(argv=None)` with `argparse.ArgumentParser().add_argument('--config', required=True)`.

</details>

<details><summary>Hint 2</summary>

End the file with `if __name__ == '__main__': main()`. `python -m medlab.evaluate` runs it as a module, so `from medlab.index import ...` works.

</details>


---
# Cumulative check 3 (60 min, no hints, no AI)

Implement a ranking model you haven't seen: **query likelihood with Dirichlet smoothing** (*IIR*, section 12.2; you may read that section). Rank documents by the log-probability that the document's language model generates the query:
$$\text{score}(q, d) = \sum_{t \in q} \log \frac{tf_{t,d} + \mu \, P(t \mid C)}{|d| + \mu}, \qquad P(t \mid C) = \frac{\text{total count of } t \text{ in the corpus}}{\text{total tokens in the corpus}}$$
Ignore query terms that aren't in the corpus at all. **Every** document gets a score, even without matching terms. Then evaluate it on test with your metrics, and compare it with BM25 in one sentence.

In [ ]:
import math

class QueryLikelihood:
    def __init__(self, index, mu=2000):
        raise NotImplementedError

    def search(self, q_tokens, k=100):
        raise NotImplementedError

In [ ]:
import math
ql = QueryLikelihood(index, mu=500)
q_tokens = pp(next(iter(test_q.values())))
res = ql.search(q_tokens, k=10)
cf = {t: sum(p.values()) for t, p in index.postings.items()}; total = sum(index.doc_len.values())
def ref(d):
    return sum(math.log((index.postings[t].get(d, 0) + 500 * cf[t] / total) / (index.doc_len[d] + 500)) for t in q_tokens if t in cf)
assert len(res) == 10 and all(abs(s - ref(d)) < 1e-9 for d, s in res)
assert abs(res[0][1] - max(ref(d) for d in index.doc_len)) < 1e-9
ql_run = {q: dict(ql.search(pp(t), k=100)) for q, t in test_q.items()}
print("QL nDCG@10:", round(metrics.evaluate_run(ql_run, test_qrels)["ndcg@10"], 4))
print("✅ cumulative check 3 passed")

✍️ *Time taken; QL versus BM25 in one sentence; and what μ does (think of it as a prior: Week 7).* (double-click to edit)



### Month 3 reflection
Update the tracker (the IR and NLP rows), write the Month 3 summary in `LOG.md`, push `medsearch-lab`, and **send me** the link.